# Operators and responses

Some transformations mix neighboring samples or map one grid onto another. Addition and pointwise multiplication cannot express these operations. In Prism, `model | operator` evaluates a model and then applies the transformation. This applies to generic 1D data; instrumental spectral responses are one specialization.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags

from prism.modeling import Component
from prism.modeling.operators import LinearOperatorModel

## Build a custom linear operator

A fixed matrix defines your own linear transformation without a model subclass. This three-point smoothing kernel averages neighboring samples; its boundary rows preserve a constant signal. The matrix does not normalize itself.

In [ ]:
x = np.linspace(-5, 5, 201)
matrix = diags([0.25, 0.5, 0.25], [-1, 0, 1], shape=(x.size, x.size)).tolil()
matrix[0, 0] += 0.25
matrix[-1, -1] += 0.25
smoothing = LinearOperatorModel(matrix.tocsr(), x=x, name="smoothing")

In [ ]:
def signal(x, amplitude, center, width):
    return amplitude*np.exp(-0.5*((x-center)/width)**2)


source = Component(signal, {"amplitude": 2., "center": 0., "width": 0.1}, name="signal")
transformed = source | smoothing

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(x, source(x), label="Source")
ax.plot(x, transformed(x), label="After smoothing")
ax.set(xlabel="Coordinate", ylabel="Value")
ax.legend()

The grid belongs to the operator. Prism evaluates the source there, applies the matrix, then samples the output on the requested grid. Flexible operators interpolate output values and return zero outside their native range; this is point interpolation, not integration over new bins. Analytic source derivatives pass through the same linear matrix. A nonlinear transformation requires its own evaluation/derivative contract and is not provided by this matrix constructor.

A matrix operator processes the incoming model values; its matrix may depend on
coordinates and parameters, but not on those values. A native interface for arbitrary
whole-profile transformations, including nonlinear operations, is planned separately but not yet implemented.

## Specialize to a spectral response

An instrument broadens and samples the emitted spectrum. `Dataset` can hold that response with the observation, leaving the intrinsic model separate. Different observations may then use different responses without duplicating the source physics.

In [ ]:
from astropy import units as u
from prism.modeling.lines import GaussianLines
from prism.modeling.operators import InstrumentResponse, SpectralResponse
from prism.modeling.fitting import Dataset, TRF

wave = np.linspace(4970., 5030., 601)
lines = {"name": ["OIII"], "position": [5006.8]*u.AA}
intrinsic = GaussianLines(
    lines, amplitude=2., fwhm=150., name="lines",
    bounds={"amplitude": (0., None), "fwhm": (10., 1000.), "offset": (-300., 300.)}, normalization='amplitude')
response = SpectralResponse(
    instrument=InstrumentResponse.from_fixed_fwhm(wave, 2.), wave=wave)

In [ ]:
data = Dataset(wave, np.zeros(wave.size), yerr=np.full(wave.size, 0.02), response=response)
rng = np.random.default_rng(41)
data.y = data.predict(intrinsic) + rng.normal(0., data.yerr)
result = TRF()(intrinsic, data)
result.params

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(wave, data.y, color="0.7", label="Observation")
ax.plot(wave, result.model(wave), label="Intrinsic fitted model")
ax.plot(wave, result.predict(), label="Through the response")
ax.set(xlabel="Wavelength [Angstrom]", ylabel="Flux density")
ax.legend()

For explicit composition, `intrinsic | response` applies the same operator in the model expression. Do not also attach that response to the Dataset. The [Joint Fitting](Joint%20Fitting.ipynb) example associates different responses with different observations.

## Local instrumental width instead

For Gaussian instrumental broadening of individual lines, `instfwhm` is a compact alternative. It accepts a constant velocity FWHM or a position-dependent table. It belongs to the line profile rather than transforming an arbitrary source spectrum; it is not a general response matrix. Use one route for the same broadening.

In [ ]:
local = GaussianLines(lines, amplitude=2., fwhm=150., instfwhm=120., name="local", normalization='amplitude')
local